# 05 · Pipelines de Preprocesamiento

> **Objetivo:** dejar listo un pipeline de scikit-learn que transforme
> el dataset por cliente en una matriz numérica lista para clusterizar.

## ¿Qué vamos a hacer?

1. Construir un pipeline numérico (imputación → log → escalado).
2. Construir un pipeline categórico (imputación → one-hot).
3. Combinarlos con `ColumnTransformer`.
4. Agregar PCA al final como opción.
5. Comparar el efecto de **no escalar** vs escalar.

## Concepto teórico: ¿por qué pipelines?

Un `Pipeline` de scikit-learn encadena transformaciones. Las ventajas son:

1. **Reproducibilidad**: el mismo objeto aplica las mismas transformaciones.
2. **Sin data leakage**: en `fit_transform(train)` el escalador aprende
   parámetros del train; al aplicar a test usa esos mismos parámetros.
3. **Serialización**: puedes guardar el pipeline entero en un `.joblib`
   y cargarlo en producción (ej. la app de Streamlit).

`ColumnTransformer` permite aplicar diferentes pipelines a diferentes
columnas: numéricas a uno, categóricas a otro, etc.

## ¿Por qué escalar es CRÍTICO para K-Means y DBSCAN?

Ambos algoritmos usan **distancia euclidiana**. Si tienes:

- `Frequency` ∈ [1, 100]
- `Monetary` ∈ [0, 200000]

entonces la distancia entre dos clientes está dominada casi 100% por
`Monetary`. `Frequency` se vuelve invisible. Escalar pone todas las
variables en el mismo rango y deja que el algoritmo "vea" todas
las dimensiones.


In [ ]:
# Permite importar el paquete src/ desde el notebook
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import pandas as pd
import numpy as np

from src.config import FEATURES_DATA_FILE, RFM_FEATURES, EXTENDED_NUMERIC_FEATURES
from src.features.preprocessing import build_preprocessing_pipeline


In [ ]:
features = pd.read_parquet(FEATURES_DATA_FILE)
features.head()


## 1. Pipeline RFM básico (numérico únicamente)

Variables: `Recency`, `Frequency`, `Monetary`. Imputación → log → escalado.


In [ ]:
pipe_rfm = build_preprocessing_pipeline(
    numeric_features=RFM_FEATURES,
    categorical_features=None,
    use_log=True,
    n_components_pca=None,
)
X_rfm = pipe_rfm.fit_transform(features[RFM_FEATURES])
print(f"Shape post-pipeline: {X_rfm.shape}")
print(f"Media:  {X_rfm.mean(axis=0).round(3)}")
print(f"Stddev: {X_rfm.std(axis=0).round(3)}")


> **Tras `StandardScaler`** la media es ~0 y la desviación ~1. Eso es
> exactamente lo que queremos para K-Means.

## 2. Pipeline extendido (RFM + features adicionales)


In [ ]:
pipe_ext = build_preprocessing_pipeline(
    numeric_features=EXTENDED_NUMERIC_FEATURES,
    use_log=True,
)
X_ext = pipe_ext.fit_transform(features[EXTENDED_NUMERIC_FEATURES])
print(f"Shape: {X_ext.shape}")


## 3. Pipeline con PCA (para visualización 2D)

PCA proyecta los datos a un subespacio de menor dimensión preservando
la mayor varianza posible. Útil para visualizar y, opcionalmente,
para reducir colinealidad antes de clusterizar.


In [ ]:
pipe_pca = build_preprocessing_pipeline(
    numeric_features=EXTENDED_NUMERIC_FEATURES,
    use_log=True,
    n_components_pca=2,
)
X_pca = pipe_pca.fit_transform(features[EXTENDED_NUMERIC_FEATURES])
print(f"Shape post-PCA: {X_pca.shape}")

pca_step = pipe_pca.named_steps["pca"]
print(f"Varianza explicada por componente: {pca_step.explained_variance_ratio_.round(3)}")
print(f"Total: {pca_step.explained_variance_ratio_.sum():.2%}")


> Si las dos primeras componentes ya capturan >80% de varianza,
> visualizar en 2D es razonable. Si no, considera 3D o usa t-SNE/UMAP.

## 4. Pipeline con variables categóricas

Si quisiéramos incluir el `Country` agrupado, podríamos hacerlo así.
(Ejemplo ilustrativo: en este proyecto principal usaremos solo numéricas.)


In [ ]:
# Ejemplo: agrupar países en categorías
features_cat = features.copy()
# Necesitaríamos joinar con la tabla original para obtener country.
# Aquí solo mostramos la estructura del pipeline.

pipe_mix = build_preprocessing_pipeline(
    numeric_features=RFM_FEATURES,
    categorical_features=None,  # cambia por ['CountryGroup'] si lo agregas
    use_log=True,
)
print(pipe_mix)


## 5. Demostración del efecto del escalado

Veamos qué pasa con K-Means **sin** escalar vs **con** escalar.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_raw = features[RFM_FEATURES].values
labels_raw = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X_raw)
sil_raw = silhouette_score(X_raw, labels_raw)

labels_scaled = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X_rfm)
sil_scaled = silhouette_score(X_rfm, labels_scaled)

print(f"Sin escalar:      Silhouette = {sil_raw:.3f}")
print(f"Con log+escalar:  Silhouette = {sil_scaled:.3f}")


> **Tip de evaluación:** el Silhouette suele subir bastante (a veces 2-3x)
> tras escalar correctamente. Es la prueba más rápida de que el pipeline
> está haciendo su trabajo.

## 6. Cuándo usar cada escalador

| Escalador | Cuándo |
|---|---|
| `StandardScaler` | Default. Centra en 0 y escala a varianza 1. |
| `MinMaxScaler` | Cuando necesitas valores en [0, 1] (ej. visualizaciones, NN). |
| `RobustScaler` | Cuando hay outliers que no quieres transformar (usa mediana e IQR). |
| `MaxAbsScaler` | Cuando los datos son sparse y quieres mantener el cero. |

## 7. Guardar el pipeline para uso posterior


In [ ]:
import joblib
from src.config import PIPELINE_FILE

joblib.dump(pipe_ext, PIPELINE_FILE)
print(f"Pipeline guardado en: {PIPELINE_FILE}")


## Resumen

- `Pipeline` → secuencia ordenada de transformaciones.
- `ColumnTransformer` → diferentes transformaciones por columna.
- Para RFM: imputación → `log1p` → `StandardScaler`.
- Escalar es **crítico** para K-Means y DBSCAN.

---

## Preguntas de Reflexión

1. ¿Por qué un `MinMaxScaler` puede ser mala idea cuando hay outliers?
2. ¿Qué pasaría si entrenaras el `StandardScaler` sobre `train + test`?
3. ¿En qué situación un PCA *empeora* el clustering?
4. Si quisieras dar más peso a `Monetary` en el clustering, ¿cómo lo harías
   sin romper el pipeline?

> **Próximo paso:** ``06_clustering_kmeans.ipynb`` — al fin clusterizamos.
